# Module 1 — Programme Orientation & Development Environment

**Hands-on objective:** Validate the local Python/Jupyter setup, connect securely to the approved Azure model endpoint, execute the first LLM call, and inspect the operational metadata returned with every model interaction.

## Environment validation

**What this demonstrates:** Confirm the notebook is running on the intended Python interpreter and version.

**What to observe:** The executable path should point to the approved training environment, and the Python version should match the standardized setup.


In [1]:
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)

Python executable: c:\Users\SrikanthRachakulla\anaconda3\envs\zs_ai\python.exe
Python version: 3.10.0 | packaged by conda-forge | (default, Nov 10 2021, 13:20:59) [MSC v.1916 64 bit (AMD64)]


## Library validation

**What this demonstrates:** Verify that the core SDKs required for Azure-hosted LLM access are available in the active environment.

**What to observe:** Successful imports confirm the local engineering stack is ready for model integration.


In [2]:
import openai
import dotenv
import azure.identity

print("openai:", openai.__version__)
print("Required libraries imported successfully")

openai: 2.53.0
Required libraries imported successfully


## Secure configuration loading

**What this demonstrates:** Load endpoint, credential, and deployment settings from the `.env` file instead of hard-coding secrets in the notebook.

**What to observe:** The notebook should confirm that configuration values exist without exposing the API key itself.


In [3]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
model = os.getenv("AZURE_OPENAI_MODEL")

print("Endpoint configured:", bool(endpoint))
print("API key configured :", bool(api_key))
print("Model configured   :", bool(model))
print("Model deployment   :", model)

Endpoint configured: True
API key configured : True
Model configured   : True
Model deployment   : gpt-4.1-mini


## Azure model client initialization

**What this demonstrates:** Create the reusable client object that connects this notebook to the approved Azure-hosted model endpoint.

**What to observe:** A successful client creation proves the SDK and endpoint configuration are syntactically valid before we send a prompt.


In [4]:
from openai import OpenAI

client = OpenAI(
    base_url=endpoint,
    api_key=api_key
)

print("Azure model client created successfully")

Azure model client created successfully


## First LLM interaction

**What this demonstrates:** Send a healthcare payer question to the deployed Azure model and retrieve the generated response.

**What to observe:** Observe how a simple user message becomes a structured API request and returns an assistant message.


In [5]:
response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": "Explain prior authorization in healthcare in two sentences."
        }
    ]
)

print(response.choices[0].message.content)

Prior authorization in healthcare is a process where healthcare providers must obtain approval from a patient's insurance company before delivering certain medical services, treatments, or medications to ensure they are medically necessary. This step helps control costs and prevent unnecessary or inappropriate care.


## Inspect the response object

**What this demonstrates:** Look beyond the generated text and inspect the full `ChatCompletion` object returned by the SDK.

**What to observe:** Notice model version, choices, finish reason, usage metadata, safety results, and latency details in addition to the answer itself.


In [6]:
print(type(response))
print(response)


<class 'openai.types.chat.chat_completion.ChatCompletion'>
ChatCompletion(id='chatcmpl-EBPOPVPnCNLsaelGGMQGhdJ0xWURT', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Prior authorization in healthcare is a process where healthcare providers must obtain approval from a patient's insurance company before delivering certain medical services, treatments, or medications to ensure they are medically necessary. This step helps control costs and prevent unnecessary or inappropriate care.", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_results={'hate': {'filtered': False, 'severity': 'safe'}, 'protected_material_code': {'detected': False, 'filtered': False}, 'protected_material_text': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'

** A production AI call is not just text generation; it is an observable transaction with metadata that can be governed, measured, and audited.


## Token usage inspection

**What this demonstrates:** Measure how many tokens were consumed by the input prompt and generated completion.

**What to observe:** Prompt, completion, and total token counts are the basic unit for understanding model consumption.


In [7]:
print("Prompt tokens     :", response.usage.prompt_tokens)
print("Completion tokens :", response.usage.completion_tokens)
print("Total tokens      :", response.usage.total_tokens)

Prompt tokens     : 16
Completion tokens : 48
Total tokens      : 64


**Takeaway:** Every extra instruction, document chunk, or generated paragraph has a measurable cost footprint — AI architecture is also token architecture.


## Safety metadata inspection

**What this demonstrates:** Inspect the content-safety signals returned with the Azure model response.

**What to observe:** The response includes moderation-related metadata for both prompt and generated content.


In [8]:
print("Prompt filter results:")
print(response.prompt_filter_results)

print("\nResponse content filter results:")
print(response.choices[0].content_filter_results)

Prompt filter results:
[{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}]

Response content filter results:
{'hate': {'filtered': False, 'severity': 'safe'}, 'protected_material_code': {'detected': False, 'filtered': False}, 'protected_material_text': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}


**Takeaway:** The model answer is only one layer; enterprise platforms can also attach policy and safety controls around each interaction.


## Latency observability

**What this demonstrates:** Inspect timing metadata such as total duration and time-to-first-token.

**What to observe:** Latency metrics reveal how quickly the model begins responding and how long the full transaction takes.


In [9]:
print(response.usage.latency_checkpoint)

{'engine_tbt_ms': 7, 'engine_ttft_ms': 27, 'engine_ttlt_ms': 358, 'pre_inference_ms': 224, 'service_tbt_ms': 7, 'service_ttft_ms': 504, 'service_ttlt_ms': 827, 'total_duration_ms': 612, 'user_visible_ttft_ms': 279}


**Takeaway:** Two models can produce equally good answers but create very different user experiences — latency is a product metric, not just an infrastructure metric.


## System prompt control

**What this demonstrates:** Add a system instruction to define domain, audience, and response behaviour.

**What to observe:** Compare the output with the earlier generic response and notice the stronger payer terminology and domain framing.


In [10]:
response_with_system = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": "You are a healthcare payer domain assistant. Explain concepts accurately and concisely for technology professionals."
        },
        {
            "role": "user",
            "content": "Explain prior authorization."
        }
    ]
)

print(response_with_system.choices[0].message.content)

Prior authorization is a utilization management process used by health insurance plans to review and approve certain medical services, procedures, or medications before they are provided. Its purpose is to ensure that the prescribed service is medically necessary, appropriate, and cost-effective according to the insurer’s coverage policies. 

Key points:  
- **Initiated by provider or patient:** Before delivering a specific treatment or drug, the healthcare provider submits a request to the insurer.  
- **Review process:** The insurer evaluates clinical information against predefined guidelines and criteria.  
- **Approval required:** Only if approved will the insurer cover the cost; otherwise, the patient may be responsible.  
- **Common in:** Expensive medications, specialty drugs, advanced imaging, elective surgeries, or treatments with alternative options.  

In technology systems, prior authorization involves workflows for request submission, documentation handling, decision track

**Takeaway:** Same model, same question, different system context — behaviour changes without retraining the model.


## System prompt control

**What this demonstrates:** Add a system instruction to define domain, audience, and response behaviour.

**What to observe:** Compare the output with the earlier generic response and notice the stronger payer terminology and domain framing.


In [11]:
print("WITHOUT SYSTEM MESSAGE")
print("-" * 50)
print(response.choices[0].message.content)

print("\nWITH SYSTEM MESSAGE")
print("-" * 50)
print(response_with_system.choices[0].message.content)

WITHOUT SYSTEM MESSAGE
--------------------------------------------------
Prior authorization in healthcare is a process where healthcare providers must obtain approval from a patient's insurance company before delivering certain treatments, medications, or services. This ensures that the proposed care is medically necessary and covered under the patient's insurance plan.

WITH SYSTEM MESSAGE
--------------------------------------------------
Prior authorization (PA) is a utilization management process used by health insurance payers to determine if a prescribed medical service, procedure, or medication is medically necessary before it is provided. The provider must obtain approval from the payer by submitting clinical information supporting the need for the service. This helps control costs, reduce unnecessary care, and ensure treatments align with evidence-based guidelines. If the PA is denied, the patient may be responsible for the cost unless an appeal overturns the decision.


**Takeaway:** Same model, same question, different system context — behaviour changes without retraining the model.


## Controlled output prompting

**What this demonstrates:** Add explicit formatting and length constraints to make the response more predictable.

**What to observe:** The model should follow the requested bullet count and concision while preserving the healthcare payer context.


In [11]:
response_controlled = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": """
You are a healthcare payer domain assistant.

Rules:
1. Answer for a technology professional.
2. Use healthcare payer terminology.
3. Keep the response to exactly 3 bullet points.
4. Do not exceed 80 words.
"""
        },
        {
            "role": "user",
            "content": "Explain prior authorization."
        }
    ]
)

print(response_controlled.choices[0].message.content)

- Prior authorization is a payer-controlled process requiring provider approval before specific services or medications are covered.  
- It ensures medical necessity, prevents fraud, and manages costs by verifying treatments align with clinical guidelines.  
- Payers use electronic portals or claim edits to enforce authorization, avoiding payment denials during claims adjudication.


**Takeaway:** Good prompting is not about making answers longer — it is about reducing variability and making outputs easier to consume downstream.


## Controlled output prompting

**What this demonstrates:** Add explicit formatting and length constraints to make the response more predictable.

**What to observe:** The model should follow the requested bullet count and concision while preserving the healthcare payer context.


In [12]:
print("Prompt tokens     :", response_controlled.usage.prompt_tokens)
print("Completion tokens :", response_controlled.usage.completion_tokens)
print("Total tokens      :", response_controlled.usage.total_tokens)

Prompt tokens     : 62
Completion tokens : 64
Total tokens      : 126


**Takeaway:** Good prompting is not about making answers longer — it is about reducing variability and making outputs easier to consume downstream.


## Controlled output prompting

**What this demonstrates:** Add explicit formatting and length constraints to make the response more predictable.

**What to observe:** The model should follow the requested bullet count and concision while preserving the healthcare payer context.


In [13]:
simple_tokens = response.usage.total_tokens
controlled_tokens = response_controlled.usage.total_tokens

increase = controlled_tokens - simple_tokens
increase_pct = (increase / simple_tokens) * 100

print("Simple prompt tokens     :", simple_tokens)
print("Controlled prompt tokens :", controlled_tokens)
print("Additional tokens        :", increase)
print(f"Increase                  : {increase_pct:.1f}%")

Simple prompt tokens     : 64
Controlled prompt tokens : 126
Additional tokens        : 62
Increase                  : 96.9%


## Few-shot prompting

**What this demonstrates:** Show the model examples of the desired response pattern before asking the target question.

**What to observe:** The final answer should mirror the terminology and compact format demonstrated in the examples.


In [14]:
few_shot_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": "You are a healthcare payer assistant. Follow the response style shown in the examples."
        },
        {
            "role": "user",
            "content": "What is a deductible?"
        },
        {
            "role": "assistant",
            "content": "Deductible: The amount a member pays out-of-pocket before the health plan begins paying for covered services."
        },
        {
            "role": "user",
            "content": "What is coinsurance?"
        },
        {
            "role": "assistant",
            "content": "Coinsurance: The percentage of an allowed healthcare cost that a member pays after meeting the deductible."
        },
        {
            "role": "user",
            "content": "What is prior authorization?"
        }
    ]
)

print(few_shot_response.choices[0].message.content)

Prior Authorization: A requirement that certain medical services, medications, or procedures must be approved by the health plan before they are provided to ensure they are medically necessary.
